In [1]:
import sys, neo4j, pandas
print(sys.version)
print(neo4j.__version__)

3.10.22 (main, Oct  9 2026, 15:18:58) [MSC v.1944 64 bit (AMD64)]
4.2.1


In [1]:
import neo4j
print(neo4j.__version__)

6.4.1


In [ ]:
from neo4j import GraphDatabase

URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "PASSWORD")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("Terhubung!")

Terhubung!


Check data

In [3]:
import pandas as pd

df = pd.read_csv("data/events.csv")
print(df.shape)
print(df.dtypes)
df.head()

(2756101, 5)
timestamp          int64
visitorid          int64
event             object
itemid             int64
transactionid    float64
dtype: object


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [4]:
df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms")
print(df["datetime"].min(), "->", df["datetime"].max())
print(df["event"].value_counts())

2015-05-03 03:00:04.384000 -> 2015-09-18 02:59:47.788000
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64


In [5]:
with driver.session(database="neo4j") as session:
    print(session.run("RETURN 1 AS ok").single()["ok"])

1


Menentukan rentang tanggal

In [7]:
P1_START, P1_END = "2015-05-12", "2015-06-26"   # END eksklusif
P2_START, P2_END = "2015-07-06", "2015-08-10"   # END eksklusif

df = pd.read_csv("data/events.csv")
df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms")

def ambil_periode(df, start, end):
    mask = (df["datetime"] >= start) & (df["datetime"] < end)
    return df[mask].copy()

p1 = ambil_periode(df, P1_START, P1_END)
p2 = ambil_periode(df, P2_START, P2_END)
p1["period"] = "P1"
p2["period"] = "P2"

print("P1:", p1.shape, p1["event"].value_counts().to_dict())
print("P2:", p2.shape, p2["event"].value_counts().to_dict())
print("Visitor P1:", p1["visitorid"].nunique(), "| Item P1:", p1["itemid"].nunique())
print("Visitor di P1 dan P2:", len(set(p1["visitorid"]) & set(p2["visitorid"])))

p1.drop(columns="datetime").to_csv("data/events_p1.csv", index=False)
p2.drop(columns="datetime").to_csv("data/events_p2.csv", index=False)
pd.concat([p1, p2]).drop(columns="datetime").to_csv("data/events_filtered.csv", index=False)
print("File tersimpan.")

P1: (940430, 7) {'view': 909730, 'addtocart': 23038, 'transaction': 7662}
P2: (762421, 7) {'view': 736710, 'addtocart': 19376, 'transaction': 6335}
Visitor P1: 479153 | Item P1: 145405
Visitor di P1 dan P2: 16174
File tersimpan.


In [8]:
#check jumlah baris
p1 = pd.read_csv("data/events_p1.csv")
p2 = pd.read_csv("data/events_p2.csv")
print(len(p1), len(p2))
print("Visitor di P1 dan P2:", len(set(p1["visitorid"]) & set(p2["visitorid"])))

940430 762421
Visitor di P1 dan P2: 16174


In [ ]:
import pandas as pd
from tqdm import tqdm
from neo4j import GraphDatabase

URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", "PASSWORD")
CSV = "data/events_p1.csv"
BATCH_SIZE = 5000

def create_constraints(session):
    session.run("CREATE CONSTRAINT visitor_id IF NOT EXISTS FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE")
    session.run("CREATE CONSTRAINT item_id IF NOT EXISTS FOR (i:Item) REQUIRE i.itemId IS UNIQUE")

def import_batch(tx, batch):
    q = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})
    WITH v, i, row
    CALL {
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)
        UNION ALL
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)
        UNION ALL
        WITH v, i, row
        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp, transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(q, batch=batch)

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()

total_rows = sum(1 for _ in open(CSV)) - 1
total_chunks = total_rows // BATCH_SIZE + 1
print("Total baris:", total_rows)

with driver.session() as session:
    create_constraints(session)
    for chunk in tqdm(pd.read_csv(CSV, chunksize=BATCH_SIZE), total=total_chunks, desc="Impor"):
        chunk = chunk[["timestamp", "visitorid", "event", "itemid", "transactionid"]].copy()
        chunk["transactionid"] = chunk["transactionid"].apply(
            lambda x: "" if pd.isna(x) else str(int(x))
        )
        session.execute_write(import_batch, chunk.to_dict(orient="records"))

driver.close()
print("Proses impor selesai.")

Total baris: 940430


Impor: 100%|██████████| 189/189 [01:35<00:00,  1.98it/s]

Proses impor selesai.
